# Notebook 1: Data Preprocessing

---

**Author**: Laura Steel Pascual  
**Date**: February 2026  
**Project**: Decoding Anthropogenic Impact with eDNA and Supervised Machine Learning: Species‑Level Associations from Solomon Island Rivers

---

## Overview
This notebook loads, cleans, and prepares data for machine learning, producing feature matrices (*X*) and binary labels (*y*) for model training.

**Purpose**: 
- Document the data processing pipeline and create model-ready datasets.

**Paper Reference**: 2. Materials & Methods -> 2.1 eDNA Sampling, 2.2 Site Classification, 2.3 Data Processing and Preprocessing

---

## 1. Setup

In [30]:
# Import libraries (install if necessary, !pip install [library])
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.preprocessing import StandardScaler, FunctionTransformer

# Create output directories
os.makedirs('results/preprocessed', exist_ok=True)

print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')

Pandas version: 2.3.2
NumPy version: 1.24.4


## 2. Load Data

Load eDNA species counts (*X* features) and LUI-derived site classifications (*y* labels).

In [32]:
# Define data directories
DATA_DIR = 'data'
PREPROCESSED_DIR = 'results/preprocessed'

# Load eDNA species data
EDNA_SPECIES = pd.read_excel(
    os.path.join(DATA_DIR, 'EDNA_AGGREGATED_SITE_SPECIES_ALL_AUGUST2024.xlsx')
)

# Load binary classifications from Notebook 1a
IMPACT_CLASS = pd.read_excel(
    os.path.join(PREPROCESSED_DIR, 'impact_classification_output.xlsx')
)

# Create working copies and set index
species_sites = EDNA_SPECIES.copy().set_index('SiteID')
impact_binary = IMPACT_CLASS.copy().set_index('SiteID')

print(f'Loaded eDNA data: {species_sites.shape}')
print('\neDNA data preview:')
display(species_sites.head())
display(species_sites.describe())

print(f'\nLoaded site classifications: {impact_binary.shape}')
print('\nSite classifications preview:')
display(impact_binary.head())
display(impact_binary.describe())

# Verify order preserved
print("\nFirst 5 sites:", species_sites.index[:5].tolist())
print("Expected: ['Mataniko_1.1', 'Mataniko_1.2', ...]")

Loaded eDNA data: (47, 862)

eDNA data preview:


,Short_SiteID,Sicyopterus cynocephalus or lagocephalus,Areca catechu,Rhyacichthys guilberti,Stiphodon semoni,Kleinhovia hospita,Kuhlia marginata,Ludwigia sp. C-006,Crenimugil cf. heterocheilos,Sicyopterus stiphodonoides,...,fungal sp. ZH G1-1,Austracris guttulosa,Hydrillodes dimissalis,Pezizales sp.,Acaulopage dichotoma,Pilidium septatum,Apocyclops sp. 1 CR-2017,Helicohyalinum infundibulum,Lejeunea stenodentata,Zasmidium pearceae
SiteID,,,,,,,,,,,,,,,,,,,,,
Mataniko_1.1,1_1,28344,1165,26839,5390,143,4819,0,1462,10016,...,0,0,0,0,0,0,0,0,0,0
Mataniko_1.2,1_2,20864,3472,20177,2147,0,15284,0,1979,6850,...,0,0,0,0,0,0,0,0,0,0
Mataniko_1.3,1_3,11467,13188,18204,2753,13339,50391,0,19364,578,...,0,5,0,0,0,0,0,0,0,5
Mataniko_1.4,1_4,8138,26222,5057,2209,10325,43070,0,7751,702,...,0,0,0,0,0,0,0,0,0,0
Mataniko_1.5,1_5,2771,106631,785,19,11322,17411,0,1581,0,...,0,0,0,0,0,5,0,0,0,0


,Sicyopterus cynocephalus or lagocephalus,Areca catechu,Rhyacichthys guilberti,Stiphodon semoni,Kleinhovia hospita,Kuhlia marginata,Ludwigia sp. C-006,Crenimugil cf. heterocheilos,Sicyopterus stiphodonoides,Belobranchus belobranchus,...,fungal sp. ZH G1-1,Austracris guttulosa,Hydrillodes dimissalis,Pezizales sp.,Acaulopage dichotoma,Pilidium septatum,Apocyclops sp. 1 CR-2017,Helicohyalinum infundibulum,Lejeunea stenodentata,Zasmidium pearceae
count,47.000000,47.000000,47.000000,47.000000,47.000000,47.000000,47.000000,47.000000,47.000000,47.000000,...,47.000000,47.000000,47.000000,47.000000,47.000000,47.000000,47.000000,47.000000,47.000000,47.000000
mean,45772.191489,32720.617021,23025.531915,16153.851064,12693.936170,12569.531915,9531.574468,9145.042553,7741.936170,6762.936170,...,0.106383,0.106383,0.106383,0.106383,0.106383,0.106383,0.106383,0.106383,0.106383,0.106383
std,40450.829698,57552.188535,21116.633798,22728.092239,39865.520604,14442.828032,46647.946825,10353.858550,30255.220108,7971.321961,...,0.729325,0.729325,0.729325,0.729325,0.729325,0.729325,0.729325,0.729325,0.729325,0.729325
min,0.000000,0.000000,430.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,18350.000000,4183.000000,8086.000000,1037.500000,0.000000,2946.000000,0.000000,343.000000,129.500000,1070.500000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,35834.000000,13188.000000,19077.000000,5749.000000,1047.000000,5884.000000,0.000000,4818.000000,1079.000000,2726.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,63674.500000,32438.500000,27886.500000,21181.000000,5263.000000,17632.000000,0.000000,19092.000000,4506.500000,10925.500000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,175295.000000,273915.000000,81808.000000,102255.000000,248569.000000,62829.000000,281028.000000,32574.000000,205559.000000,33879.000000,...,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000



Loaded site classifications: (47, 8)

Site classifications preview:


,Impacted_100m_0.03125,Impacted_100m_0.0625,Impacted_300m_0.03125,Impacted_300m_0.0625,Impacted_500m_0.03125,Impacted_500m_0.0625,Impacted_1000m_0.03125,Impacted_1000m_0.0625
SiteID,,,,,,,,
Charigaveheu_8.2,0,0,0,0,0,0,0,0
Charigaveheu_8.3,0,0,1,0,0,0,1,0
Charimalenga_7.0,0,0,0,0,0,0,0,0
Charimalenga_7.1,0,0,0,0,0,0,0,0
Charimalenga_7.2,0,0,0,0,1,0,0,0


,Impacted_100m_0.03125,Impacted_100m_0.0625,Impacted_300m_0.03125,Impacted_300m_0.0625,Impacted_500m_0.03125,Impacted_500m_0.0625,Impacted_1000m_0.03125,Impacted_1000m_0.0625
count,47.000000,47.000000,47.000000,47.000000,47.000000,47.000000,47.000000,47.000000
mean,0.446809,0.425532,0.531915,0.468085,0.659574,0.510638,0.617021,0.382979
std,0.502538,0.499769,0.504375,0.504375,0.478975,0.505291,0.491369,0.491369
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,1.000000,0.000000,1.000000,1.000000,1.000000,0.000000
75%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000



First 5 sites: ['Mataniko_1.1', 'Mataniko_1.2', 'Mataniko_1.3', 'Mataniko_1.4', 'Mataniko_1.5']
Expected: ['Mataniko_1.1', 'Mataniko_1.2', ...]


## 3. Binary Site Classifications

### 3.1 Overview

**Site classifications**, *y* labels (from Notebook 1a):
- Binary labels: 0 = reference, 1 = impacted.
- 8 buffer-threshold combinations: 4 buffers (100m, 300m, 500m, 1000m) × 2 thresholds (0.03125, 0.0625).

Binary classifications reflect maximum Land Use Intensification (LUI) scores across 2017-2023 (excluding 2021) for each buffer distance.

**Threshold selection** (see Notebook 1a for details):
- 0.03125 (1/32 pastureland).
- 0.0625 (1/16 pastureland).

These values balance class distribution (30-62% impacted) with ecological relevance for Guadalcanal's largely pristine catchments.

### 3.2 Class Balance

In [34]:
# Display class balance for all 8 combinations
buffers = [100, 300, 500, 1000]
thresholds = [0.03125, 0.0625]

print("\nClass balance across buffer-threshold combinations:")
for thresh in thresholds:
    print(f"\nThreshold: {thresh}")
    for buf in buffers:
        col = f'Impacted_{buf}m_{thresh}'
        n_impacted = impact_binary[col].sum()
        n_reference = len(impact_binary) - n_impacted
        pct_impacted = (n_impacted / len(impact_binary)) * 100
        print(f'  {buf}m: {n_reference} reference ({100-pct_impacted:.1f}%), '
              f'{n_impacted} impacted ({pct_impacted:.1f}%)')



Class balance across buffer-threshold combinations:

Threshold: 0.03125
  100m: 26 reference (55.3%), 21 impacted (44.7%)
  300m: 22 reference (46.8%), 25 impacted (53.2%)
  500m: 16 reference (34.0%), 31 impacted (66.0%)
  1000m: 18 reference (38.3%), 29 impacted (61.7%)

Threshold: 0.0625
  100m: 27 reference (57.4%), 20 impacted (42.6%)
  300m: 25 reference (53.2%), 22 impacted (46.8%)
  500m: 23 reference (48.9%), 24 impacted (51.1%)
  1000m: 29 reference (61.7%), 18 impacted (38.3%)


## 4. eDNA Species Data

### 4.1 Data Origins

**Collection and processing**:
- eDNA samples collected across 8 rivers (47 sites) on Guadalcanal, Solomon Islands, August 2024.
- Metabarcoding by Wilderlab NZ using COI barcode.
- Bioinformatic processing: species-level read assignment.
- Quality control by DDA International Consulting Ltd:
  - Metadata cleaning.
  - Control sample removal.
  - Absent taxa filtering.
  - Replicate aggregation to site-level counts.

**Result**: 47 sites × 861 species (*X* features), with read counts ranging from 0 to ~281,000.

### 4.2 Extract Feature Matrix

**Note**: Large variance across species (orders of magnitude difference: some ~281,000 reads, others <10) and many zeros. 

In [36]:
# Identify species columns (exclude SiteIDShort_SiteID, SiteID is now the index)
species_cols = [col for col in species_sites.columns 
                if col not in ['Short_SiteID']]
                
# Verify first few species look biological
print("First 5 species columns:", species_cols[:5])

# Clean column names
species_cols_cleaned = [col.replace('[', '_').replace(']', '_')
                           .replace('<', '_').replace('>', '_')
                        for col in species_cols]

# Extract raw counts and site identifiers
X_raw = species_sites[species_cols].values  # RAW COUNTS (not transformed)
site_ids = species_sites.index.values       # use index (SiteID) as IDs

print(f'Feature matrix: {X_raw.shape}')
print(f'  Sites: {len(site_ids)}')
print(f'  Species: {len(species_cols)}')
print(f'\nSite ID order preserved: {site_ids[0]} (should be Mataniko_1.1)')
print(f'\nCount distribution:')
print(f'  Range: {X_raw.min()} to {X_raw.max()}')
print(f'  Mean: {X_raw.mean():.2f}')
print(f'  Median: {np.median(X_raw):.2f}')

# Save raw counts (X features) with cleaned column names
X_raw_df = pd.DataFrame(X_raw, columns=species_cols_cleaned)
X_raw_df.insert(0, 'SiteID', site_ids)
X_raw_df.to_excel('results/preprocessed/X_raw.xlsx', index=False)

print('Saved X_raw.xlsx')


First 5 species columns: ['Sicyopterus cynocephalus or lagocephalus', 'Areca catechu', 'Rhyacichthys guilberti', 'Stiphodon semoni', 'Kleinhovia hospita']
Feature matrix: (47, 861)
  Sites: 47
  Species: 861

Site ID order preserved: Mataniko_1.1 (should be Mataniko_1.1)

Count distribution:
  Range: 0 to 281028
  Mean: 343.22
  Median: 0.00
Saved X_raw.xlsx


### 4.3 Transformation Rationale

Species counts exhibit:
1. Extreme variance: Abundant species vs rare species.
2. Right-skewed distributions: Most values near zero, long tail of high counts.
3. Scale differences: ML algorithms can perform poorly when features span vastly different ranges.

However, rare taxa need to be retained as potential indicators of environmental condition.

**Solution**: Sequential transformation
- Log(x+1): Compresses variance, reduces outlier influence.
- Z-score standardisation: Centres features (mean=0, std=1).

### 4.4 Transformation Strategy

Transformations occur **within ML pipelines** (Notebook 2), not here. This prevents data leakage:

- Scaler parameters (mean, std) estimated from training folds only.
- Test folds transformed using training fold parameters.
- No information flows from test to training data.

Pipeline implementation example (Notebook 2):
```python
Pipeline([
    ('log_transform', FunctionTransformer(np.log1p)),    # Step 1
    ('scaler', StandardScaler()),                        # Step 2: fit on train only
    ('classifier', RandomForestClassifier())             # Step 3
])
```

### 4.5 Transformation Preview

Create transformed data for manual inspection (not used in modelling).

**Note**: This preview demonstrates the transformation effect but is not used in modelling. The actual transformations occur in Notebook 2 pipelines.

In [38]:
# Apply transformations
X_log = np.log1p(X_raw)  # Log(x+1)
X_scaled = StandardScaler().fit_transform(X_log)  # Z-score

print('Raw counts:')
print(f'  Range: {X_raw.min():.0f} to {X_raw.max():.0f}')
print(f'  Mean: {X_raw.mean():.2f}')

print('\nLog-transformed:')
print(f'  Range: {X_log.min():.3f} to {X_log.max():.3f}')
print(f'  Mean: {X_log.mean():.3f}')

print('\nLog + Standardized:')
print(f'  Range: {X_scaled.min():.3f} to {X_scaled.max():.3f}')
print(f'  Mean: {X_scaled.mean():.3f} (expect ~0)')
print(f'  Std: {X_scaled.std():.3f} (expect ~1)')

# Save preview for inspection
transformed_preview = pd.DataFrame(X_scaled, columns=species_cols)
transformed_preview.insert(0, 'SiteID', site_ids)
transformed_preview.to_excel('results/preprocessed/X_transformed_preview.xlsx', index=False)

print('\nSaved transformed preview')


Raw counts:
  Range: 0 to 281028
  Mean: 343.22

Log-transformed:
  Range: 0.000 to 12.546
  Mean: 0.692

Log + Standardized:
  Range: -4.535 to 6.782
  Mean: -0.000 (expect ~0)
  Std: 1.000 (expect ~1)

Saved transformed preview


## 5. Export Preprocessed Datasets

### 5.1 Save Model-Ready Data (Pickle Files)

Export 8 datasets (4 buffers × 2 thresholds) combining raw counts (*X* features) with binary labels (*y* variable).

In [40]:
# impact_binary is already aligned and indexed by SiteID
impact_binary_aligned = impact_binary.loc[site_ids]

# Export each combination
for buf in buffers:
    for thresh in thresholds:
        # Extract labels
        label_col = f'Impacted_{buf}m_{thresh}'
        y = impact_binary_aligned[label_col].values
        
        # Package data
        data = {
            'X_raw': X_raw,                # Raw counts (NOT transformed)
            'y': y,                        # Binary labels (0=ref, 1=impact)
            'feature_names': species_cols, # Species names (original)
            'site_ids': site_ids,          # Site identifiers (FIXED)
            'buffer': buf,
            'threshold': thresh,
            'n_sites': len(y),
            'n_species': len(species_cols),
            'n_reference': (y == 0).sum(),
            'n_impacted': (y == 1).sum()
        }
        
        # Save
        thresh_str = str(thresh).replace('.', '')
        filename = f'data_{buf}m_t{thresh_str}.pkl'
        filepath = os.path.join('results/preprocessed', filename)
        
        with open(filepath, 'wb') as f:
            pickle.dump(data, f)
        
        print(f'{filename}: {data["n_reference"]} reference, {data["n_impacted"]} impacted')

print('\nSaved 8 preprocessed datasets')


data_100m_t003125.pkl: 26 reference, 21 impacted
data_100m_t00625.pkl: 27 reference, 20 impacted
data_300m_t003125.pkl: 22 reference, 25 impacted
data_300m_t00625.pkl: 25 reference, 22 impacted
data_500m_t003125.pkl: 16 reference, 31 impacted
data_500m_t00625.pkl: 23 reference, 24 impacted
data_1000m_t003125.pkl: 18 reference, 29 impacted
data_1000m_t00625.pkl: 29 reference, 18 impacted

Saved 8 preprocessed datasets


### 5.2 Save Species List

In [42]:
# Export species list for reference
pd.DataFrame({
    'species_id': range(len(species_cols)),
    'species_name': species_cols
}).to_csv('results/preprocessed/feature_names.csv', index=False)

print('Saved species list')


Saved species list


## 6. Summary

**Outputs** (`results/preprocessed/`):
- **Model inputs** (8 pickle files): `data_100m_t003125.pkl` to `data_1000m_t00625.pkl`
- **Reference files**: `feature_names.csv`
- **Inspection files**: `X_raw.xlsx`, `X_transformed_preview.xlsx`

**Each pickle contains**:
- `X_raw`: Raw species counts (47 × 861, **not transformed**)
- `y`: Binary labels (47 × 1, 0=reference, 1=impacted)
- `feature_names`: Species list (861 names)
- `site_ids`: Site identifiers
- Metadata: buffer distance, LUI threshold, class counts

### Next Steps
- Notebook 2: `2_Model_Training.ipynb`
- Notebook 2a: `2a_CV_Fold_Comparison.ipynb`
- Notebook 2b: `2b_Binary_vs_Counts_Comparison.ipynb`

### Alternative Approach: Binary Presence/Absence

- Alternative tested: Converting counts to binary (1=present, 0=absent).
- Rationale: Simplifies data, potentially reduces sequencing depth bias.
- Decision: Count data was selected after comparison (see Notebook 2c for full evaluation).